In [1]:
def compute_lp_diff(initial_rank, final_rank, wl):
    if ((initial_rank[0] == final_rank[0]) & (initial_rank[1] == final_rank[1])):
        rank_difference = final_rank[2] - initial_rank[2]
    else:
        if wl:
            rank_difference = -(initial_rank[2] + (100-final_rank[2]))
        else:
            rank_difference = (100-initial_rank[2] + final_rank[2])
    
    return rank_difference

In [5]:
initial = ["PLATINUM", 1, 16]
final = ["PLATINUM", 2, 95]

compute_lp_diff(initial, final, True)

-21

In [4]:
import requests

api_key = "RGAPI-2bb4eee0-806d-4e90-96a4-06f8e5a53ea4"
puuid = "im5SFOKnFNPFn0K3y9WjZQrCTHOkhe3_4ejZC33A63S0j6LFgkxq1yh7A2sQf5SZoTEmsWboQy9m0A"

# 1. Broad query (no queue filter)
url = f"https://americas.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?start=0&count=10"
headers = {"X-Riot-Token": api_key}

response = requests.get(url, headers=headers)
print("Response:", response.json())

Response: ['NA1_5624064395', 'NA1_5624050150', 'NA1_5624029055', 'NA1_5623996281', 'NA1_5623989028', 'NA1_5623948180', 'NA1_5623938988', 'NA1_5623916901', 'NA1_5623905750', 'NA1_5623888035']


In [7]:
def get_account_info(puuid):
    platform_host = "na1"

    league_v4_url = (
        f"https://{platform_host}.api.riotgames.com/"
        f"lol/league/v4/entries/by-puuid/{puuid}"
    )

    response = requests.get(league_v4_url, headers=headers)
    response.raise_for_status()

    lv4_response = response.json()

    solo_queue = next(
        (
            entry
            for entry in lv4_response
            if entry["queueType"] == "RANKED_SOLO_5x5"
        ),
        None
    )

    if solo_queue is None:
        raise ValueError(
            "No RANKED_SOLO_5x5 entry found for this account."
        )

    latest_match_id = get_latest_match_id(puuid)

    return solo_queue, latest_match_id

In [ ]:
def get_latest_match_id(puuid):
    """
    Gets the most recent Ranked Solo/Duo match only.
    """

    matches_url = (
        "https://americas.api.riotgames.com/"
        f"lol/match/v5/matches/by-puuid/{puuid}/ids"
        f"?queue=420&count=1"
    )

    response = requests.get(matches_url, headers=headers)
    response.raise_for_status()
    retry_after = int(response.headers.get("Retry-After", 1))
    print(f"Rate limited! Sleeping for {retry_after} seconds...")

    matches = response.json()

    if not matches:
        return None

    return matches[0]

In [9]:
get_latest_match_id(puuid)

HTTPError: 429 Client Error: Too Many Requests for url: https://americas.api.riotgames.com/lol/match/v5/matches/by-puuid/im5SFOKnFNPFn0K3y9WjZQrCTHOkhe3_4ejZC33A63S0j6LFgkxq1yh7A2sQf5SZoTEmsWboQy9m0A/ids?queue=420&count=1

In [8]:
get_account_info(puuid)

HTTPError: 429 Client Error: Too Many Requests for url: https://americas.api.riotgames.com/lol/match/v5/matches/by-puuid/im5SFOKnFNPFn0K3y9WjZQrCTHOkhe3_4ejZC33A63S0j6LFgkxq1yh7A2sQf5SZoTEmsWboQy9m0A/ids?queue=420&count=1